In [3]:
import os
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.impute import SimpleImputer


sns.set(style="whitegrid")

In [13]:

def load_and_clean(path, target_col):
    df = pd.read_csv(path)
    df = df.copy()

    # Lowercase column names
    df.columns = [c.strip() for c in df.columns]

    # Basic cleaning: drop exact duplicate rows
    if df.duplicated().sum() > 0:
        print(f"Dropping {df.duplicated().sum()} duplicate rows")
        df = df.drop_duplicates().reset_index(drop=True)

    # If target is Yes/No or string, convert to 0/1
    if df[target_col].dtype == object or str(df[target_col].dtype).startswith('category'):
        df[target_col] = df[target_col].astype(str).str.strip().str.lower()
        # common mappings
        df[target_col] = df[target_col].replace({"yes": 1, "no": 0, "y": 1, "n": 0, "true": 1, "false": 0, "1": 1, "0": 0})

    # Try to coerce to numeric
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')

    if df[target_col].isna().sum() > 0:
        raise ValueError(f"Target column {target_col} contains non-convertible values after coercion.\n" \
                         f"Please inspect the dataset. NaNs: {df[target_col].isna().sum()}")

    return df

In [25]:
def build_preprocessor(df, target_col):
    # Identify categorical and numerical features
    X = df.drop(columns=[target_col])
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    # Imputers and transformers
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

    return preprocessor, numeric_cols, categorical_cols


def get_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc = roc_auc_score(y_true, y_proba)
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, roc_auc=roc)

In [15]:

def plot_confusion_matrix(y_true, y_pred, out_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()


def plot_roc_curve(y_true, y_proba, out_path):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,6))
    plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0,1], [0,1], linestyle='--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(out_path, bbox_inches='tight')
    plt.close()


def baseline_random_forest(X_train, X_test, y_train, y_test, preprocessor, outputs_dir):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('clf', RandomForestClassifier(random_state=42))])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:,1]

    metrics = get_metrics(y_test, y_pred, y_proba)

    # Save plots
    plot_confusion_matrix(y_test, y_pred, os.path.join(outputs_dir, 'confusion_matrix_baseline.png'))
    plot_roc_curve(y_test, y_proba, os.path.join(outputs_dir, 'roc_curve_baseline.png'))

    return pipeline, metrics

In [16]:


def cross_validation_compare(X, y, preprocessor, k=5):
    # Build pipeline
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    scoring = 'roc_auc'

    skf_scores = cross_val_score(pipeline, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    kf_scores = cross_val_score(pipeline, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    return dict(stratified_mean=skf_scores.mean(), stratified_std=skf_scores.std(),
                nonstrat_mean=kf_scores.mean(), nonstrat_std=kf_scores.std(),
                stratified_scores=skf_scores, nonstrat_scores=kf_scores)


def hyperparameter_tuning(X_train, y_train, preprocessor, outputs_dir):
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('clf', RandomForestClassifier(random_state=42))])

    param_grid = {
        'clf__n_estimators': [50, 100, 200, 400],
        'clf__max_depth': [None, 5, 10, 20]
    }

    grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
    grid.fit(X_train, y_train)

    # Prepare heatmap data
    results = pd.DataFrame(grid.cv_results_)
    # pivot: mean test score across params
    results['param_clf__max_depth'] = results['param_clf__max_depth'].astype(str)
    heat_df = results.pivot_table(index='param_clf__n_estimators', columns='param_clf__max_depth', values='mean_test_score')

    plt.figure(figsize=(8,6))
    sns.heatmap(heat_df, annot=True, fmt='.4f')
    plt.title('GridSearchCV mean_test_score')
    plt.xlabel('max_depth')
    plt.ylabel('n_estimators')
    plt.savefig(os.path.join(outputs_dir, 'gridsearch_heatmap.png'), bbox_inches='tight')
    plt.close()

    # RandomizedSearch
    from scipy.stats import randint
    param_dist = {
        'clf__n_estimators': randint(50, 500),
        'clf__max_depth': [None] + list(range(3, 31))
    }
    rand = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=20, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1)
    rand.fit(X_train, y_train)

    return dict(grid_best_params=grid.best_params_, grid_best_score=grid.best_score_,
                rand_best_params=rand.best_params_, rand_best_score=rand.best_score_, grid=grid, rand=rand)



In [17]:




def evaluate_final(model_pipeline, X_test, y_test, outputs_dir):
    y_pred = model_pipeline.predict(X_test)
    y_proba = model_pipeline.predict_proba(X_test)[:,1]
    metrics = get_metrics(y_test, y_pred, y_proba)

    plot_confusion_matrix(y_test, y_pred, os.path.join(outputs_dir, 'confusion_matrix_final.png'))
    plot_roc_curve(y_test, y_proba, os.path.join(outputs_dir, 'roc_curve_final.png'))

    return metrics

In [27]:






def main(data_path="/home/manik/Downloads/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv",
         test_size=0.2,
         k_folds=5,
         outputs_dir="./outputs",
         target_col="Churn"
         
         ):
    os.makedirs(outputs_dir, exist_ok=True)

    print('Loading data...')
    df = load_and_clean(data_path, target_col)

    # Show class distribution
    print('\nClass distribution:')
    print(df[target_col].value_counts())

    preprocessor, num_cols, cat_cols = build_preprocessor(df, target_col)
    print(f"\nDetected numeric cols ({len(num_cols)}): {num_cols}")
    print(f"Detected categorical cols ({len(cat_cols)}): {cat_cols}")

    X = df.drop(columns=[target_col])
    y = df[target_col].astype(int)

    # Train-test split (stratify to keep class distribution)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    print(f"\nTrain/Test sizes: {X_train.shape[0]} / {X_test.shape[0]}")

    print('\n1) Baseline Random Forest (default params)')
    baseline_pipe, baseline_metrics = baseline_random_forest(X_train, X_test, y_train, y_test, preprocessor, outputs_dir)
    print('Baseline metrics:')
    print(baseline_metrics)

    print('\n2) Cross-validation comparison (Stratified KFold vs KFold)')
    cv_results = cross_validation_compare(X, y, preprocessor, k=k_folds)
    print('CV Results:')
    print(f"Stratified: mean={cv_results['stratified_mean']:.4f}, std={cv_results['stratified_std']:.4f}")
    print(f"Non-stratified: mean={cv_results['nonstrat_mean']:.4f}, std={cv_results['nonstrat_std']:.4f}")

    print('\n3) Hyperparameter tuning (GridSearchCV & RandomizedSearchCV)')
    tuning = hyperparameter_tuning(X_train, y_train, preprocessor, outputs_dir)
    print('GridSearch best params:', tuning['grid_best_params'])
    print('GridSearch best score:', tuning['grid_best_score'])
    print('RandomizedSearch best params:', tuning['rand_best_params'])
    print('RandomizedSearch best score:', tuning['rand_best_score'])

    # Use best estimator from randomized search for final evaluation
    final_pipeline = tuning['rand'].best_estimator_
    final_metrics = evaluate_final(final_pipeline, X_test, y_test, outputs_dir)
    print('\nFinal evaluation metrics:')
    print(final_metrics)

    # Summarize metrics table
    metrics_df = pd.DataFrame([baseline_metrics, final_metrics], index=['baseline', 'final'])
    print('\nMetrics summary table:')
    print(metrics_df)
    metrics_df.to_csv(os.path.join(outputs_dir, 'metrics_summary.csv'))

    # Also show feature importance for the RandomForest after preprocessing
    try:
        # Extract feature names after preprocessing
        preproc = final_pipeline.named_steps['preprocessor']
        clf = final_pipeline.named_steps['clf']

        # numeric names
        num_names = num_cols
        # categorical names from OneHotEncoder
        cat_names = []
        if len(cat_cols) > 0:
            ohe = preproc.named_transformers_['cat'].named_steps['onehot']
            cat_feature_names = ohe.get_feature_names_out(cat_cols)
            cat_names = list(cat_feature_names)
        feature_names = list(num_names) + cat_names

        importances = clf.feature_importances_
        fi = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(30)
        plt.figure(figsize=(8,10))
        fi.plot(kind='barh')
        plt.gca().invert_yaxis()
        plt.title('Top 30 Feature Importances')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'feature_importances.png'))
        plt.close()
        print(f"Saved feature importance plot to {outputs_dir}")
    except Exception as e:
        print('Could not compute feature importances after preprocessing:', e)

    print('\nAll outputs saved to:', outputs_dir)






In [28]:
main()

Loading data...

Class distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Detected numeric cols (3): ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Detected categorical cols (17): ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']

Train/Test sizes: 5634 / 1409

1) Baseline Random Forest (default params)


/tmp/ipykernel_6851/3532114867.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[target_col] = df[target_col].replace({"yes": 1, "no": 0, "y": 1, "n": 0, "true": 1, "false": 0, "1": 1, "0": 0})


Baseline metrics:
{'accuracy': 0.794889992902768, 'precision': 0.6534296028880866, 'recall': 0.4839572192513369, 'f1': 0.5560675883256528, 'roc_auc': 0.820843989769821}

2) Cross-validation comparison (Stratified KFold vs KFold)
CV Results:
Stratified: mean=0.8272, std=0.0126
Non-stratified: mean=0.8270, std=0.0110

3) Hyperparameter tuning (GridSearchCV & RandomizedSearchCV)
Fitting 3 folds for each of 16 candidates, totalling 48 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
GridSearch best params: {'clf__max_depth': 20, 'clf__n_estimators': 100}
GridSearch best score: 0.8354155988372794
RandomizedSearch best params: {'clf__max_depth': 29, 'clf__n_estimators': 326}
RandomizedSearch best score: 0.8379649229628621

Final evaluation metrics:
{'accuracy': 0.7856635911994322, 'precision': 0.7045454545454546, 'recall': 0.3315508021390374, 'f1': 0.4509090909090909, 'roc_auc': 0.8267289260895399}

Metrics summary table:
          accuracy  precision    recall        f1   r